In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-10'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # ckpt = 20000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  # ckpt = 20000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.0548, -0.3166,  0.0779,  0.6623, -1.3936,  0.0758, -0.0302,  0.1652,
         -0.1108,  0.0225, -0.9296, -0.0144]], device='cuda:0')
Scaled actions :  tensor([[-0.0548, -0.3166,  0.0779,  0.6623, -1.3936,  0.0758, -0.0302,  0.1652,
         -0.1108,  0.0225, -0.9296, -0.0144]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0829e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05, -5.4768e-02, -3.1662e-01,
          7.7881e-02,  6.6233e-01, -1.3936e+00,  7.5800e-02, -3.0228e-02,
          1.6519e-01, -1.1081e-01,  2.2543e-02, -9.2960e-01, -1.4403e-02]],
       device='cuda:0')
torques: [ 3.95066936e-16 -9.12378428e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.17744398e-16  2.14534117e-16 -6.60884875e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.92077407e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.1685, -0.5723,  0.3545,  0.5973, -1.9714, -0.3297,  0.0652,  0.5515,
         -0.3805,  0.3063, -1.5953,  0.1093]], device='cuda:0')
Scaled actions :  tensor([[-0.1685, -0.5723,  0.3545,  0.5973, -1.9714, -0.3297,  0.0652,  0.5515,
         -0.3805,  0.3063, -1.5953,  0.1093]], device='cuda:0')
obs :  tensor([[-9.2774e-03, -5.5145e-02, -8.6021e-02, -1.0502e-03, -1.4422e-04,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.0789e-02,
         -7.8759e-03, -1.8476e-03,  2.4297e-02, -1.0954e-01,  2.1785e-02,
         -9.1135e-03,  6.4111e-03, -9.9131e-03,  2.0796e-02, -1.0115e-01,
         -6.3535e-03, -5.1172e-02, -6.4424e-02,  3.4857e-03,  1.8810e-01,
         -9.9169e-01,  8.6941e-02, -4.2773e-02,  5.7081e-02, -6.4748e-02,
          1.5428e-01, -8.9552e-01, -4.5520e-02, -1.6854e-01, -5.7225e-01,
          3.5452e-01,  5.9725e-01, -1.9714e+00, -3.2975e-01,  6.5223e-02,
          5.5147e-01, -3.8046e-01,  3.0634e-01, -1.5953e+00,  1.0

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.0469, -0.4503,  0.2138, -0.0611, -0.7482, -0.2964,  0.1094,  0.1501,
          0.4651, -0.2426, -0.2075,  0.2164]], device='cuda:0')
Scaled actions :  tensor([[-0.0469, -0.4503,  0.2138, -0.0611, -0.7482, -0.2964,  0.1094,  0.1501,
          0.4651, -0.2426, -0.2075,  0.2164]], device='cuda:0')
obs :  tensor([[ 4.2354e-02, -1.2146e-01, -2.6690e-01, -4.7500e-03, -1.1187e-03,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -4.9347e-02,
         -2.6592e-02,  2.9570e-03,  7.7264e-02, -4.1669e-01, -5.5039e-02,
          6.7857e-03,  2.0765e-02, -4.0512e-02,  8.3768e-02, -3.8566e-01,
          2.6782e-02, -2.5270e-01, -1.2711e-01,  4.2010e-02,  3.2072e-01,
         -1.9798e+00, -5.8098e-01,  1.1030e-01,  9.6631e-02, -2.1463e-01,
          4.1902e-01, -1.8519e+00,  1.7425e-01, -4.6933e-02, -4.5032e-01,
          2.1384e-01, -6.1108e-02, -7.4823e-01, -2.9640e-01,  1.0939e-01,
          1.5005e-01,  4.6509e-01, -2.4257e-01, -2.0753e-01,  2.1

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.2252,  0.2086, -0.5649, -0.8089,  0.3013, -0.3163, -0.5017,  0.4028,
          0.3138, -0.9174,  0.7604,  0.4052]], device='cuda:0')
Scaled actions :  tensor([[ 0.2252,  0.2086, -0.5649, -0.8089,  0.3013, -0.3163, -0.5017,  0.4028,
          0.3138, -0.9174,  0.7604,  0.4052]], device='cuda:0')
obs :  tensor([[ 0.1173, -0.6214, -0.4483, -0.0218, -0.0041, -0.9998,  1.0000,  0.0000,
          0.0000, -0.0619, -0.0665,  0.0506,  0.1041, -0.7077, -0.1392,  0.0378,
          0.0440, -0.0465,  0.1312, -0.6513,  0.0886,  0.0204, -0.2579,  0.3615,
         -0.0044, -1.0236, -0.3402,  0.1450,  0.1262,  0.1077,  0.0959, -0.9018,
          0.2771,  0.2252,  0.2086, -0.5649, -0.8089,  0.3013, -0.3163, -0.5017,
          0.4028,  0.3138, -0.9174,  0.7604,  0.4052]], device='cuda:0')
torques: [  15.44253744 -200.           -2.82045334 -200.          200.
   26.84406632   -2.12070609   91.69334417  200.         -200.
  200.          -22.08207296]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.4082,  1.0107, -1.0760,  0.1253, -0.1680,  0.1991,  0.3374, -1.0572,
         -1.1059, -0.4813, -0.2593, -0.3244]], device='cuda:0')
Scaled actions :  tensor([[-0.4082,  1.0107, -1.0760,  0.1253, -0.1680,  0.1991,  0.3374, -1.0572,
         -1.1059, -0.4813, -0.2593, -0.3244]], device='cuda:0')
obs :  tensor([[-4.9169e-01, -5.5113e-01, -1.9201e-01, -4.5003e-02,  4.1768e-03,
         -9.9898e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.7635e-03,
         -8.8851e-02,  1.2167e-01,  8.9780e-02, -8.0474e-01, -2.0046e-01,
          7.6255e-03,  9.9758e-02, -4.8089e-04,  1.1082e-01, -7.2752e-01,
          1.8843e-01,  4.2288e-01,  1.7930e-02,  3.4699e-01, -1.2184e-01,
         -4.5312e-02, -2.5648e-01, -4.0274e-01,  3.9782e-01,  3.4500e-01,
         -2.6556e-01,  4.6080e-02,  4.6712e-01, -4.0817e-01,  1.0107e+00,
         -1.0760e+00,  1.2530e-01, -1.6799e-01,  1.9905e-01,  3.3740e-01,
         -1.0572e+00, -1.1059e+00, -4.8132e-01, -2.5929e-01, -3.2

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-0.6772, -0.7748, -0.6814,  0.1357, -0.8828, -0.2232,  0.6304, -1.3062,
         -0.3613, -0.2125, -0.6776, -0.2910]], device='cuda:0')
Scaled actions :  tensor([[-0.6772, -0.7748, -0.6814,  0.1357, -0.8828, -0.2232,  0.6304, -1.3062,
         -0.3613, -0.2125, -0.6776, -0.2910]], device='cuda:0')
obs :  tensor([[-0.6208, -0.0179, -0.0663, -0.0569,  0.0242, -0.9981,  1.0000,  0.0000,
          0.0000,  0.0136, -0.0691,  0.1449,  0.1027, -0.7218, -0.1384,  0.0036,
          0.1532,  0.0175,  0.1041, -0.6503,  0.1246, -0.1067,  0.1278, -0.0366,
          0.1079,  0.8427,  0.5978,  0.2295,  0.2494, -0.0626, -0.0511,  0.6464,
         -0.7154, -0.6772, -0.7748, -0.6814,  0.1357, -0.8828, -0.2232,  0.6304,
         -1.3062, -0.3613, -0.2125, -0.6776, -0.2910]], device='cuda:0')
torques: [-200.          200.         -200.           53.16516378  200.
  190.73580877  200.         -200.         -200.         -200.
  200.         -200.        ]
データ収集: step 7


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.3896,  0.0036,  0.5805, -0.4587, -1.1880, -0.5001, -1.1264, -0.2676,
          0.8295, -0.9398, -0.0572,  0.3482]], device='cuda:0')
Scaled actions :  tensor([[ 0.3896,  0.0036,  0.5805, -0.4587, -1.1880, -0.5001, -1.1264, -0.2676,
          0.8295, -0.9398, -0.0572,  0.3482]], device='cuda:0')
obs :  tensor([[-0.1257,  0.6259, -0.0818, -0.0435,  0.0403, -0.9982,  1.0000,  0.0000,
          0.0000, -0.0700, -0.0686,  0.1090,  0.1207, -0.6581, -0.1215,  0.1047,
          0.1788, -0.0307,  0.0965, -0.6187, -0.0167, -0.6690, -0.0887, -0.3027,
          0.0857, -0.1099, -0.2194,  0.7308,  0.0119, -0.3838, -0.0115, -0.1382,
         -0.6036,  0.3896,  0.0036,  0.5805, -0.4587, -1.1880, -0.5001, -1.1264,
         -0.2676,  0.8295, -0.9398, -0.0572,  0.3482]], device='cuda:0')
torques: [-200.         -200.         -200.          -54.84710537 -200.
  -36.60230021  200.         -200.         -200.         -200.
 -126.67387641   55.75074024]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.8435,  0.0015,  1.1032,  1.0323, -0.9066,  0.3763, -0.9417,  0.0622,
          1.2481, -0.2686, -0.0150, -0.1238]], device='cuda:0')
Scaled actions :  tensor([[ 0.8435,  0.0015,  1.1032,  1.0323, -0.9066,  0.3763, -0.9417,  0.0622,
          1.2481, -0.2686, -0.0150, -0.1238]], device='cuda:0')
obs :  tensor([[-9.2805e-02, -1.1838e-03, -1.4006e-01, -3.2563e-02,  4.4164e-02,
         -9.9849e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.4529e-01,
         -7.3885e-02,  9.4132e-02,  1.2562e-01, -7.5957e-01, -2.0937e-01,
          1.8720e-01,  1.8170e-01, -7.1120e-02,  9.7326e-02, -5.4713e-01,
         -4.1479e-02, -1.4201e-01,  1.5085e-02,  1.0940e-01, -2.0130e-02,
         -7.0630e-01, -5.0100e-01,  2.0636e-01,  9.3918e-04, -9.8272e-02,
          1.5012e-02,  6.0811e-01,  2.3785e-01,  8.4347e-01,  1.4897e-03,
          1.1032e+00,  1.0323e+00, -9.0660e-01,  3.7630e-01, -9.4170e-01,
          6.2234e-02,  1.2481e+00, -2.6859e-01, -1.4995e-02, -1.2

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.3961, -0.1996, -1.4029, -0.2301, -0.4176,  0.0343, -0.0875, -0.2072,
         -1.0019, -0.6837, -0.0963, -0.0827]], device='cuda:0')
Scaled actions :  tensor([[-0.3961, -0.1996, -1.4029, -0.2301, -0.4176,  0.0343, -0.0875, -0.2072,
         -1.0019, -0.6837, -0.0963, -0.0827]], device='cuda:0')
obs :  tensor([[-0.1251, -0.5468, -0.1335, -0.0447,  0.0489, -0.9978,  1.0000,  0.0000,
          0.0000, -0.1140, -0.0662,  0.1409,  0.1320, -0.9330, -0.2012,  0.1737,
          0.1891, -0.0588,  0.0853, -0.3754, -0.0628,  0.4004,  0.0578,  0.3311,
          0.0753, -0.6265,  0.4847, -0.2911,  0.0557,  0.1973, -0.1209,  1.0070,
         -0.3146, -0.3961, -0.1996, -1.4029, -0.2301, -0.4176,  0.0343, -0.0875,
         -0.2072, -1.0019, -0.6837, -0.0963, -0.0827]], device='cuda:0')
torques: [ 200.           92.55807278  200.          200.          200.
  200.         -200.         -200.          200.         -200.
  200.         -144.17231285]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.8696, -0.6142, -0.3183, -0.6428, -0.0628, -0.6920,  0.2622, -0.1221,
          0.3776, -0.0654, -0.7519,  0.3716]], device='cuda:0')
Scaled actions :  tensor([[-0.8696, -0.6142, -0.3183, -0.6428, -0.0628, -0.6920,  0.2622, -0.1221,
          0.3776, -0.0654, -0.7519,  0.3716]], device='cuda:0')
obs :  tensor([[ 0.4140,  0.0922,  0.2665, -0.0525,  0.0416, -0.9978,  1.0000,  0.0000,
          0.0000, -0.1001, -0.0807,  0.1785,  0.1278, -0.9517, -0.1035,  0.0996,
          0.1739, -0.0477,  0.0709, -0.2141, -0.0801, -0.1956, -0.1715,  0.0683,
         -0.0952,  0.3419,  0.3644, -0.3998, -0.1713, -0.0431, -0.0489,  0.6690,
         -0.0200, -0.8696, -0.6142, -0.3183, -0.6428, -0.0628, -0.6920,  0.2622,
         -0.1221,  0.3776, -0.0654, -0.7519,  0.3716]], device='cuda:0')
torques: [-200.          -91.4115449  -200.         -200.          200.
 -146.81704348   11.50074231 -200.         -200.         -200.
 -200.           10.59208005]
データ収集: step 1

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.8516,  1.7358,  0.0828, -0.3329, -0.5486, -0.0227,  0.5627,  1.4226,
          0.1532,  0.6424, -0.3504, -0.0518]], device='cuda:0')
Scaled actions :  tensor([[ 0.8516,  1.7358,  0.0828, -0.3329, -0.5486, -0.0227,  0.5627,  1.4226,
          0.1532,  0.6424, -0.3504, -0.0518]], device='cuda:0')
obs :  tensor([[ 9.4068e-01, -3.0573e-02,  3.3228e-01, -5.0716e-02,  1.3885e-02,
         -9.9862e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -2.0610e-01,
         -1.4469e-01,  1.8409e-01,  8.6270e-02, -7.7621e-01, -1.3827e-01,
          5.3383e-02,  1.1685e-01, -5.0978e-02,  7.8451e-02, -1.6162e-01,
          1.1516e-04, -8.1041e-01, -4.3422e-01, -2.0425e-02, -3.0399e-01,
          1.3161e+00, -6.1469e-01, -8.0463e-02, -3.7903e-01,  5.2746e-03,
          1.0079e-01, -8.5410e-02,  7.0928e-01,  8.5161e-01,  1.7358e+00,
          8.2836e-02, -3.3295e-01, -5.4858e-01, -2.2667e-02,  5.6268e-01,
          1.4226e+00,  1.5316e-01,  6.4235e-01, -3.5041e-01, -5.

In [46]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=3.827, Scaled action max=3.827
Step 1/10, Total steps: 232
steps: 232
actions : tensor([[-2.0073e-01,  2.7012e-03, -3.9236e+00,  1.0792e+00, -3.2072e+00,
         -5.9513e-02, -1.2939e-01, -1.2166e+00,  2.3700e+00, -4.3837e+00,
          3.8273e+00,  1.6591e-01]], device='cuda:0')
target_dof_pos: tensor([[ 5.6819e-01,  1.7057e-01, -3.2847e+00,  3.0802e+00, -2.5938e+00,
         -4.6042e-02, -1.7757e-03,  4.2487e-03,  1.0755e+00, -2.9165e+00,
          4.7486e-01,  8.3404e-01]], device='cuda:0')
Step 1: Original action max=5.134, Scaled action max=5.134
Step 2: Original action max=4.670, Scaled action max=4.670
データ収集完了: 10 steps collected with action_scale=1.0


In [47]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [48]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
